In [2]:
import os
import json
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()


EDAMAM_FOOD_APP_ID = os.getenv("EDAMAM_FOOD_APP_ID")
EDAMAM_FOOD_APP_KEY = os.getenv("EDAMAM_FOOD_APP_KEY")


In [3]:



# API endpoint for nutrients
base_url_nutrients = 'https://api.edamam.com/api/food-database/v2/nutrients'

# Cache file
nutrition_lookup_file = 'nutrition_lookup.json'

# Load cache
def load_nutrition_cache():
    if os.path.exists(nutrition_lookup_file):
        with open(nutrition_lookup_file, 'r') as f:
            return json.load(f)
    return {}

# Save cache
def save_nutrition_cache(nutrition_lookup):
    with open(nutrition_lookup_file, 'w') as f:
        json.dump(nutrition_lookup, f, indent=2)

# Get nutrient data for a food ID
def get_nutrient_data(food_id, quantity=100, measure="gram"):
    nutrition_lookup = load_nutrition_cache()
    
    # Check cache first
    cache_key = f"{food_id}_{quantity}_{measure}"
    if cache_key in nutrition_lookup:
        return nutrition_lookup[cache_key]
    
    # Prepare API request
    payload = {
        "ingredients": [{
            "quantity": quantity,
            "measureURI": f"http://www.edamam.com/ontologies/edamam.owl#Measure_{measure}",
            "foodId": food_id
        }]
    }
    params = {
        'app_id': EDAMAM_FOOD_APP_ID,
        'app_key': EDAMAM_FOOD_APP_KEY
    }
    
    # Make API request
    try:
        response = requests.post(base_url_nutrients, params=params, json=payload)
        response.raise_for_status()
        data = response.json()
        
        # Cache result
        nutrition_lookup[cache_key] = data
        save_nutrition_cache(nutrition_lookup)
        
        return data
    except requests.exceptions.RequestException as e:
        print(f"Error getting nutrient data: {e}")
        return None

# # Example usage
# if __name__ == "__main__":
#     # Example food_id for whole milk
#     food_id = "food_b49rs1kaw0jktabzkg2vvanvvsis"
#     nutrient_data = get_nutrient_data(food_id)
    
#     if nutrient_data:
#         print(f"Calories: {nutrient_data['totalNutrients'].get('ENERC_KCAL', {}).get('quantity')} kcal")
#         print(f"Protein: {nutrient_data['totalNutrients'].get('PROCNT', {}).get('quantity')} g")
#         print(f"Fat: {nutrient_data['totalNutrients'].get('FAT', {}).get('quantity')} g")

In [7]:
import os
import json
import time
from pathlib import Path

gemini_edamam_dir = os.path.abspath("gemini_edamam/")
gemini_edamam_files = [f for f in os.listdir(gemini_edamam_dir) if f.endswith(".json")]

outdir = "gemini_edamam_nutrition"
os.makedirs(outdir, exist_ok=True)

for file in gemini_edamam_files:
    path = os.path.join(gemini_edamam_dir, file)
    out_path = os.path.join(outdir, file)
    
    # Check if file exists
    if not os.path.exists(path):
        print(f"File {path} does not exist. Skipping.")
        continue

    if os.path.exists(out_path):
        print(f"Skipping {file}. Output file exists. ")
        continue
        
    with open(path, 'r') as f:
        data = json.load(f)
        food_id = data['most_relevant_id']
        nutrient_data = get_nutrient_data(food_id)

        
        with open(out_path, 'w') as f:
            json.dump(nutrient_data, f, indent=2)

        print(f"Saved nutrient data for {file}")
    
        time.sleep(0.5)


Skipping chocolate_cookie.json. Output file exists. 
Skipping green_apple_amino_energy.json. Output file exists. 
Skipping bud_light_beer.json. Output file exists. 
Skipping banana_chip.json. Output file exists. 
Skipping oyster.json. Output file exists. 
Skipping parmesan.json. Output file exists. 
Skipping rösti.json. Output file exists. 
Skipping huevos_rancheros.json. Output file exists. 
Skipping gin.json. Output file exists. 
Skipping cilantro.json. Output file exists. 
Skipping chicken_strip.json. Output file exists. 
Skipping lipton.json. Output file exists. 
Skipping swiss_cheese.json. Output file exists. 
Skipping spring_roll.json. Output file exists. 
Skipping risotto.json. Output file exists. 
Skipping atkins_shake.json. Output file exists. 
Skipping maitake_mushroom.json. Output file exists. 
Skipping sucralose.json. Output file exists. 
Skipping chana.json. Output file exists. 
Skipping pilsner.json. Output file exists. 
Skipping madeleine.json. Output file exists. 
Skipp